# Full Build Analysis

This notebook demonstrates comprehensive build-wide analysis using the Pipeline API to process all trace files in parallel.

We'll create three main DataFrames:
1. **Metadata DataFrame**: One row per build unit with compilation statistics
2. **Phase DataFrame**: Compilation phase breakdown for all build units
3. **Template DataFrame**: Template instantiation events across the entire build

All DataFrames are keyed by `build_unit` (the source file name) for easy cross-analysis.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import pandas as pd
import plotly.express as px

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from trace_analysis import (
    Pipeline,
    find_trace_files,
    parse_file,
    get_trace_file,
)
from trace_analysis.build_helpers import extract_all_data, print_phase_hierarchy

# Display settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

## Find Trace Files

In [ ]:
# Configure the path to your trace files
TRACE_DIR = Path("../../../build-trace")

json_files = find_trace_files(TRACE_DIR)

if not json_files:
    print(f"No trace files found in {TRACE_DIR}")
    print("\nTo generate trace files:")
    print("1. Configure your build with: cmake -DCMAKE_CXX_FLAGS='-ftime-trace' ...")
    print("2. Build your project")
    print("3. Trace files will be generated alongside object files")
    raise ValueError(f"No trace files found in {TRACE_DIR}")
else:
    print(f"Found {len(json_files):,} trace files")

## Parse All Files in Parallel

Parse all trace files using all available CPUs with progress tracking.

In [ ]:
# Parse all files in parallel, returning a list of raw trace dataframes.
parsed_dfs = (
    Pipeline(json_files)
    .map(parse_file, workers=-1, desc="Parsing trace files")
    .collect()
)

print(f"\nParsed {len(parsed_dfs):,} files")
print(f"Total events across all files: {sum(len(df) for df in parsed_dfs):,}")

## Create Three Analysis DataFrames

Extract metadata, phase breakdown, and template events from all parsed files in parallel.
This creates three DataFrames:
1. **metadata_df**: One row per build unit with compilation statistics
2. **phase_df**: Phase breakdown for all build units
3. **template_df**: Template instantiation events across the build

All DataFrames use a shared categorical `build_unit` column for efficient grouping and joining.

📝 **TODO:**
The details of this processing is all exposed in these notebook cells. We should add library functionality to simplify this.

📝 **TODO:**
This takes way too long, there is likely something going wrong with the in-memory dataframe processing.

In [ ]:
# Extract all three types of data in one parallel pass (this can take a few minutes)
results = (
    Pipeline(parsed_dfs)
    .map(
        extract_all_data, workers=-1, desc="Extracting metadata, phases, and templates"
    )
    .collect()
)

print(f"\nExtracted data from {len(results):,} build units")

In [ ]:
# Create shared categorical dtype for build_unit column
build_units = [r["build_unit"] for r in results]
build_unit_dtype = pd.CategoricalDtype(
    categories=sorted(set(build_units)), ordered=False
)

print(f"Created categorical dtype with {len(build_unit_dtype.categories)} categories")

In [ ]:
# Build metadata DataFrame with categorical build_unit
metadata_df = pd.DataFrame(
    [{"build_unit": r["build_unit"], **r["metadata"]} for r in results]
)
metadata_df["build_unit"] = metadata_df["build_unit"].astype(build_unit_dtype)

# Build trace file mapping and store in DataFrame attributes
trace_file_mapping = {r["build_unit"]: r["trace_file_path"] for r in results}
metadata_df.attrs["trace_file_mapping"] = trace_file_mapping

print(f"metadata_df: {metadata_df.shape[0]:,} rows (one per build unit)")
print(f"Stored trace file mapping for {len(trace_file_mapping):,} build units")

In [ ]:
# Build phase DataFrame with categorical build_unit
phase_df = pd.concat(
    [
        r["phase"].assign(
            build_unit=pd.Categorical(
                [r["build_unit"]] * len(r["phase"]), dtype=build_unit_dtype
            )
        )
        for r in results
    ],
    ignore_index=True,
)

print(f"phase_df: {phase_df.shape[0]:,} rows (phases across all build units)")

In [ ]:
# Build template DataFrame with categorical build_unit
template_df = pd.concat(
    [
        r["template"].assign(
            build_unit=pd.Categorical(
                [r["build_unit"]] * len(r["template"]), dtype=build_unit_dtype
            )
        )
        for r in results
    ],
    ignore_index=True,
)

print(f"template_df: {template_df.shape[0]:,} rows (template events across build)")

In [ ]:
# Summary of created DataFrames
print("\n=== Created Analysis DataFrames ===")
print(f"  metadata_df: {metadata_df.shape[0]:,} rows × {metadata_df.shape[1]} columns")
print(f"  phase_df: {phase_df.shape[0]:,} rows × {phase_df.shape[1]} columns")
print(f"  template_df: {template_df.shape[0]:,} rows × {template_df.shape[1]} columns")
print(
    f"\nAll DataFrames share the same categorical build_unit dtype with {len(build_unit_dtype.categories)} categories"
)

## Build-Wide Metadata Analysis

Analyze compilation statistics across all build units.

In [ ]:
# Overall build statistics
print("=== Build-Wide Statistics ===")
print(f"Total build units: {len(metadata_df):,}")
print(f"Total compilation time: {metadata_df['total_wall_time_s'].sum():.1f} seconds")
print(f"Average time per unit: {metadata_df['total_wall_time_s'].mean():.2f} seconds")
print(f"Median time per unit: {metadata_df['total_wall_time_s'].median():.2f} seconds")
print(f"Slowest unit: {metadata_df['total_wall_time_s'].max():.2f} seconds")
print(f"Fastest unit: {metadata_df['total_wall_time_s'].min():.2f} seconds")

In [ ]:
# Top 20 slowest compilation units
print("\n=== Top 20 Slowest Compilation Units ===")
slowest = metadata_df.nlargest(20, "total_wall_time_s")[
    ["build_unit", "total_wall_time_s"]
]

display(slowest.style.format({"total_wall_time_s": lambda x: f"{x:.1f}"}))

## Getting Trace Files for Interesting Build Units

Now that we've identified interesting build units (e.g., slowest compilation times), we can easily get their trace files for deeper analysis.

In [ ]:
# Example: Get trace file for a specific build unit
example_build_unit = slowest.iloc[0]["build_unit"]
trace_file = get_trace_file(metadata_df, example_build_unit)

print(f"Build unit: {example_build_unit}")
print(f"Trace file: {trace_file}")

In [ ]:
# Get trace files for all slowest compilation units
print("\n=== Trace Files for Top 10 Slowest Compilation Units ===\n")
for idx, row in slowest.head(10).iterrows():
    build_unit = row["build_unit"]
    compile_time = row["total_wall_time_s"]
    trace_file = get_trace_file(metadata_df, build_unit)
    print(f"{compile_time:6.1f}s  {build_unit}")
    print(f"         → {trace_file}\n")

In [ ]:
# Compilation time distribution
fig = px.histogram(
    metadata_df,
    x="total_wall_time_s",
    nbins=600,
    title="Distribution of Compilation Times",
    labels={"total_wall_time_s": "Compilation Time (seconds)"},
    log_y=True,
)
fig.show()

## Build-Wide Phase Analysis

Analyze compilation phases across the entire build.

In [ ]:
# Add duration_ms column for easier reading
phase_df["duration_ms"] = phase_df["duration"] / 1000.0

print(f"Total phase records: {len(phase_df):,}")
print(f"Unique phases: {phase_df['name'].nunique()}")
print(f"\nPhases tracked: {sorted(phase_df['name'].unique())}")

In [ ]:
# Cumulative phase time across entire build - hierarchical view
print_phase_hierarchy(phase_df)

In [ ]:
# Visualize cumulative phase breakdown
phase_summary = (
    phase_df.groupby(["name", "parent", "depth"]).agg({"duration": "sum"}).reset_index()
)

fig = px.sunburst(
    phase_summary,
    names="name",
    parents="parent",
    values="duration",
    title="Cumulative Phase Breakdown Across Entire Build",
    branchvalues="total",
)
fig.show()

In [ ]:
# Which build units spend most time in Frontend?
frontend_time = phase_df[phase_df["name"] == "Frontend"].nlargest(20, "duration_ms")[
    ["build_unit", "duration_ms"]
]

# Convert to seconds (keep as float)
frontend_time["duration_s"] = frontend_time["duration_ms"] / 1000
frontend_time = frontend_time[["build_unit", "duration_s"]]

print("=== Top 20 Build Units by Frontend Time ===")
display(frontend_time.style.format({"duration_s": "{:,.1f}"}))

In [ ]:
# Which build units spend most time in Backend?
backend_time = phase_df[phase_df["name"] == "Backend"].nlargest(20, "duration_ms")[
    ["build_unit", "duration_ms"]
]

backend_time["duration_s"] = backend_time["duration_ms"] / 1000
backend_time = backend_time[["build_unit", "duration_s"]]

print("=== Top 20 Build Units by Backend Time ===")
display(backend_time.style.format({"duration_s": "{:,.1f}"}))

## Build-Wide Template Analysis

Analyze template instantiations across the entire build.

In [ ]:
# Overall template statistics
print("=== Build-Wide Template Statistics ===")
print(f"Total template instantiation events: {len(template_df):,}")
print(f"Total template time: {template_df['dur'].sum() / 1_000_000:.1f} seconds")
print(f"Average template time: {template_df['dur'].mean() / 1000:.2f} ms")
print(f"Unique template names: {template_df['template_name'].nunique():,}")
print(f"Unique namespaces: {template_df['namespace'].nunique()}")

In [ ]:
# Top templates by total time across build
top_templates = template_df.groupby(["namespace", "template_name"]).agg(
    {"dur": ["count", "sum", "mean"]}
)
top_templates.columns = ["count", "total_dur", "avg_dur"]
top_templates["total_s"] = top_templates["total_dur"] / 1_000_000
top_templates["avg_ms"] = top_templates["avg_dur"] / 1000
top_templates = top_templates.sort_values("total_dur", ascending=False).reset_index()

print("\n=== Top 30 Templates by Total Time Across Build ===")
display(
    top_templates.head(30)[
        ["namespace", "template_name", "count", "total_s", "avg_ms"]
    ].style.format({"count": "{:,.0f}", "total_s": "{:,.1f}", "avg_ms": "{:,.1f}"})
)

In [ ]:
# Template time by namespace
namespace_summary = (
    template_df.groupby("namespace")
    .agg({"dur": ["count", "sum", "mean"], "param_count": "mean"})
    .round(2)
)
namespace_summary.columns = ["count", "total_dur", "avg_dur", "avg_params"]
namespace_summary["total_s"] = namespace_summary["total_dur"] / 1_000_000
namespace_summary = namespace_summary.sort_values("total_dur", ascending=False)

print("\n=== Template Time by Namespace ===")
display(
    namespace_summary.style.format(
        {
            "count": "{:,d}",
            "total_dur": "{:,.0f}",
            "avg_dur": "{:,.0f}",
            "avg_params": "{:,.2f}",
            "total_s": "{:,.1f}",
        }
    )
)

In [ ]:
# CK library templates analysis
ck_templates = template_df[template_df["is_ck_type"]].copy()

print("=== CK Library Template Analysis ===")
print(f"CK template instantiations: {len(ck_templates):,}")
print(f"CK template time: {ck_templates['dur'].sum() / 1_000_000:.1f} seconds")
print(
    f"Percentage of total template time: {100 * ck_templates['dur'].sum() / template_df['dur'].sum():.1f}%"
)

# Top CK templates
ck_by_name = (
    ck_templates.groupby("template_name")
    .agg({"dur": ["count", "sum", "mean"]})
    .round(2)
)
ck_by_name.columns = ["count", "total_dur", "avg_dur"]
ck_by_name["total_s"] = ck_by_name["total_dur"] / 1_000_000
ck_by_name = ck_by_name.sort_values("total_dur", ascending=False)

print("\n=== Top 20 CK Templates by Total Time ===")
display(
    ck_by_name.head(20)[["count", "total_s"]].style.format(
        {"count": "{:,d}", "total_s": "{:,.0f}"}
    )
)

## Cross-Analysis: Templates vs Compilation Time

Analyze relationships between template instantiations and compilation time.

In [ ]:
# Template count per build unit
template_counts = (
    template_df.groupby("build_unit").size().reset_index(name="template_count")
)

# Template time per build unit
template_time = (
    template_df.groupby("build_unit")["dur"].sum().reset_index(name="template_time_us")
)
template_time["template_time_s"] = template_time["template_time_us"] / 1_000_000

# Merge with metadata
analysis_df = (
    metadata_df[["build_unit", "total_wall_time_s"]]
    .merge(template_counts, on="build_unit", how="left")
    .merge(
        template_time[["build_unit", "template_time_s"]], on="build_unit", how="left"
    )
)

# Fill NaN with 0 for units with no templates
analysis_df["template_count"] = analysis_df["template_count"].fillna(0)
analysis_df["template_time_s"] = analysis_df["template_time_s"].fillna(0)

print("=== Template Count vs Compilation Time ===")
print(
    f"Correlation: {analysis_df['template_count'].corr(analysis_df['total_wall_time_s']):.3f}"
)

# Top units by template count
print("\n=== Top 20 Build Units by Template Count ===")
display(analysis_df.nlargest(20, "template_count"))

In [ ]:
# Scatter plot: template count vs compilation time
fig = px.scatter(
    analysis_df,
    x="template_count",
    y="total_wall_time_s",
    hover_data=["build_unit"],
    title="Template Count vs Compilation Time",
    labels={
        "template_count": "Number of Template Instantiations",
        "total_wall_time_s": "Compilation Time (seconds)",
    },
    trendline="ols",
)
fig.show()

In [ ]:
# Scatter plot: template time vs compilation time
# Note: The total instantiation double-counts nested templates.
fig = px.scatter(
    analysis_df,
    x="template_time_s",
    y="total_wall_time_s",
    hover_data=["build_unit"],
    title="Template Instantiation Time vs Total Compilation Time",
    labels={
        "template_time_s": "Template Instantiation Time (seconds)",
        "total_wall_time_s": "Total Compilation Time (seconds)",
    },
    trendline="ols",
)
fig.show()

## Summary

This notebook demonstrated:
- Parallel parsing of all trace files using the Pipeline API
- Parallel extraction of metadata, phases, and templates in a single pass
- Creating three consistently-keyed DataFrames with shared categorical dtype
- **Trace file mapping** stored in metadata_df.attrs for easy lookup
- Build-wide metadata analysis
- Cumulative phase analysis with visualizations
- Build-wide template analysis
- Cross-analysis between templates and compilation time
- **Using get_trace_file() to locate trace files for interesting build units**

The shared categorical `build_unit` dtype enables efficient grouping and joining across all three DataFrames for comprehensive build analysis.

The trace file mapping allows you to quickly download or analyze the raw trace JSON for any build unit of interest.